In [ ]:
import polars as pl
from datasets import load_dataset

In [ ]:
dataset = load_dataset(
    "SII-WANGZJ/Polymarket_data",
    data_files="quant.parquet",
    streaming=True,
    split="train"
)

In [ ]:
# Grab first 1000 rows to inspect
sample = []
for i, row in enumerate(dataset):
    sample.append(row)
    if i >= 999:
        break

In [ ]:
q_sample = pl.DataFrame(sample)

In [ ]:
markets = pl.read_parquet('data/raw/markets.parquet')

In [ ]:
political_keywords = [
    "election", "president", "congress", "senate", "minister",
    "vote", "party", "democrat", "republican", "trump", "biden",
    "harris", "political", "govern", "parliament", "prime minister",
    "geopolit", "ukraine", "israel", "nato", "war", "sanction"
]

sports_noise = ["o/u", "spread", "over/under", "moneyline", 
                "vs.", "nfl", "nba", "mlb", "nhl", "epl"]
def political_search(df):
    # Apply filter using political words
    political = df.filter(
        pl.col('closed') == 1,
        pl.any_horizontal([
            pl.col('question').str.to_lowercase().str.contains(kw)
            for kw in political_keywords
        ])
    )

    # Remove sports noise which is missed in 1st filter
    political_clean = political.filter(
        ~pl.any_horizontal([
            pl.col('question').str.to_lowercase().str.contains(kw)
            for kw in sports_noise
        ])
    )

    # Parse outcome_prices to create a resolved_yes column
    def _parse_resolved_yes(x):
        import ast
        try:
            # Handle both JSON array strings and plain strings
            x_list = ast.literal_eval(x)
            return float(x_list[0]) > 0.5
        except Exception:
            return None

    # Apply parsing function to outcome_prices column
    politicalNO_final = political_clean.with_columns(
        pl.col('outcome_prices').map_elements(
            _parse_resolved_yes,
            return_dtype=pl.Boolean
        ).alias('resolved_yes')
    )
     

    return political_final

political_df = political_search(markets)

In [ ]:
political_df.write_parquet('data/processed/markets_political.parquet')

In [ ]:
political_df.shape

In [ ]:
political_df.null_count() / len(political_df) * 100

In [ ]:
5952+14735

In [ ]:
print(political_df.filter(
    pl.col('resolved_yes').is_null()
))

In [ ]:
print(f"Ratio: {round(5952/20687, 3)}:{round(14735/20687, 3)}")

In [ ]:
pd = political_df.with_columns(
    pl.col('end_date').dt.year().alias('YYYY'),
    pl.col('end_date').dt.month().alias('MM')
)

In [ ]:
date_df = pd.group_by(['YYYY', 'MM']).agg(
    pl.len().alias('count')
).sort(['YYYY', 'MM']).with_columns(
    pl.col('YYYY').cast(pl.String).alias('YYYY'),
    pl.col('MM').cast(pl.String).alias('MM')
).with_columns(
    (pl.col('YYYY') + '-' + pl.col('MM')).alias('YYYY-MM')
).drop(['MM'])

In [ ]:
date_df

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
year_data = pd.group_by('YYYY').agg(
    pl.len().alias('count')
)

In [ ]:
year_data

In [ ]:
plt.figure(figsize=(40,6))
sns.barplot(data=date_df, x='YYYY-MM', y='count', hue='YYYY')

In [ ]:
print(political_df['resolved_yes'].value_counts())

In [ ]:
def _parse_resolved_yes(x):
    try:
        # Handle both JSON array strings and plain strings
        if x is None:
            return None
        x = x.strip()
        # Try standard JSON parse first
    

        parsed = json.loads(x)
        return float(parsed[0]) > 0.5
    except Exception:
        try:
            # Handle single-quoted or malformed strings
            x = x.replace("'", '"')
            parsed = json.loads(x)
            return float(parsed[0]) > 0.5
        except Exception:
            return None   

In [ ]:
import ast
l = political_df['outcome_prices'].to_list()
parsed_list = [ast.literal_eval(item) if isinstance(item, str) else item for item in l]

In [ ]:
ast.literal_eval(l[0])

In [ ]:
for x in political_df['outcome_prices'].to_list():
    print(x)

In [ ]:
markets.filter(
    pl.col('closed') == 1
)

In [ ]:
q_sample